In [15]:
import mwparserfromhell
import re
import json, random

def extract_yearly_events(wikitext):
    """
    Parses wikitext to extract a dictionary of {year: [list of events]}.
    """
    # Parse the raw wikitext into a wikicode object
    wikicode = mwparserfromhell.parse(wikitext)
    
    timeline_data = {}
    
    # get_sections() iterates through all sections divided by headings
    for section in wikicode.get_sections():
        headings = section.filter_headings()
        if not headings:
            continue
            
        # Extract the first heading of the section.
        # strip_code() safely handles cases where the heading is a link like == [[2001]] ==
        heading_title = headings[0].title.strip_code().strip()
        
        # Check if the heading title is exactly a 4-digit year
        if re.fullmatch(r'\d{4}', heading_title):
            year = int(heading_title)
            events = []
            
            # Split the section's wikitext into individual lines to look for bullet points
            lines = str(section).split('\n')
            for line in lines:
                line = line.strip()
                
                # Wikipedia list items start with '*' or '**'
                if line.startswith('*'):
                    # Remove the wikitext list operators and leading/trailing whitespace
                    clean_line = line.lstrip('* ').strip()
                    
                    if clean_line:
                        # Parse the single line to strip out wiki links, bolding, HTML tags, and <ref> tags
                        parsed_line = mwparserfromhell.parse(clean_line).strip_code().strip()
                        parsed_line_splitted = parsed_line.split(":")
                        to_append = "The year " + parsed_line_splitted[-1].strip()
                        
                        if to_append:
                            if not (len(parsed_line_splitted) == 2 and "-" in parsed_line_splitted[0]) \
                                and str(year) not in to_append \
                                and 0 < len(parsed_line_splitted[-1].strip().split()) <= 10:
                                events.append(to_append)
            
            # Add to our dictionary if events were found under this year
            if events:
                # If the year already exists, extend it (handles edge cases in wiki structuring)
                if year in timeline_data:
                    timeline_data[year].extend(events)
                else:
                    timeline_data[year] = events
                
    return timeline_data

In [23]:

with open("data/timeline18thcentury.txt") as f:
    raw_wikitext = f.read()
    # 2. Extract the data
    data = extract_yearly_events(raw_wikitext)

    # 3. Output the parsed data structure
    # Printing as formatted JSON for easy verification
    sampled = {k: random.choice(v) for k, v in data.items()}

In [24]:
sampled

{}